# كشكول التجارب: فوائد البحث الهجين عملياً (المحاضرة 43)

أهلاً بيك! في الكشكول ده، مش هنقعد نتكلم نظري عن مميزات البحث الهجين، إحنا هنجرب بنفسنا 3 مواقف حقيقية من الشغل:

1. **كلمة نادرة جداً أو طبية** (زي `osteoporosis`): الـ BM25 بيطير عليها فوراً بينما الفيكتور ممكن يتلخبط.
2. **سؤال بالمعنى من غير ولا كلمة مشتركة**: الفيكتور بيفهم المعنى على طول والـ BM25 بيجيب صفر.
3. **سؤال تسوق مركب** (زي `cheap noise canceling headphones`): محتاجين الفيكتور يفهم إن `cheap` معناها `affordable`، ومحتاجين الـ BM25 عشان يأكد وجود تقنية الـ `ANC`.

تعال نشوف كل موقف بعيننا!


In [1]:
# نجهز المستندات والمسترجعات الأول
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers.bm25 import BM25Retriever

try:
    from langchain.retrievers.ensemble import EnsembleRetriever
except ImportError:
    from langchain_classic.retrievers.ensemble import EnsembleRetriever

# وثائق من مجالات مختلفة
docs = [
    Document(page_content="API usage: call client.chat.completions.create(stream=True) for streaming.", metadata={"id": "doc_api"}),
    Document(page_content="Clinical diagnosis of osteoporosis requires bone mineral density T-score <= -2.5.", metadata={"id": "doc_medical"}),
    Document(page_content="Affordable ANC wireless earbuds with active noise cancellation and clear mic.", metadata={"id": "doc_ecommerce"}),
    Document(page_content="Architectural blueprint for building autonomous reasoning agent systems.", metadata={"id": "doc_ai"}),
    Document(page_content="European castles, gothic cathedrals, and historic medieval towers.", metadata={"id": "doc_travel"})
]

# تجهيز المسترجعات
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
dense_ret = vectorstore.as_retriever(search_kwargs={"k": 2})

sparse_ret = BM25Retriever.from_documents(docs)
sparse_ret.k = 2

hybrid_ret = EnsembleRetriever(retrievers=[dense_ret, sparse_ret], weights=[0.5, 0.5])

print("البيانات جاهزة!")


C:\Users\aliashraf\AppData\Local\Temp\ipykernel_5928\1057574232.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

البيانات جاهزة!


### تجربة 1: مصطلح طبي نادر
تعال نسأل عن كلمة طبية نادرة: `osteoporosis`


In [2]:
q1 = "osteoporosis bone scan"

print(f"السؤال: '{q1}'")
print("1. الكثيف طلع:", [d.metadata['id'] for d in dense_ret.invoke(q1)])
print("2. المتناثر طلع:", [d.metadata['id'] for d in sparse_ret.invoke(q1)])
print("3. الهجين طلع:", [d.metadata['id'] for d in hybrid_ret.invoke(q1)])


السؤال: 'osteoporosis bone scan'
1. الكثيف طلع: ['doc_medical', 'doc_ecommerce']
2. المتناثر طلع: ['doc_medical', 'doc_travel']
3. الهجين طلع: ['doc_medical', 'doc_ecommerce', 'doc_travel']


### تجربة 2: معنى دلالي من غير كلمات مشتركة خالص
تعال نسأل عن `smart thinking bots`.. الكلمات دي مش موجودة خالص في جملة الذكاء الاصطناعي، بس المعنى واحد!


In [3]:
q2 = "smart thinking bots"

print(f"السؤال: '{q2}'")
print("1. الكثيف طلع:", [d.metadata['id'] for d in dense_ret.invoke(q2)])
print("2. المتناثر طلع:", [d.metadata['id'] for d in sparse_ret.invoke(q2)])
print("3. الهجين طلع:", [d.metadata['id'] for d in hybrid_ret.invoke(q2)])


السؤال: 'smart thinking bots'
1. الكثيف طلع: ['doc_ai', 'doc_api']
2. المتناثر طلع: ['doc_travel', 'doc_ai']
3. الهجين طلع: ['doc_ai', 'doc_travel', 'doc_api']


### تجربة 3: سؤال التجارة الإلكترونية المركب
تعال نسأل: `cheap noise canceling headphones`.. الجملة فيها ترادف (`cheap` = `affordable`) وفيها مطابقة حرفية (`noise canceling` = `noise cancellation`):


In [4]:
q3 = "cheap noise canceling headphones"

print(f"السؤال: '{q3}'")
print("الهجين قدر يجمع بين المترادفات والحروف وطلع الوثيقة:", hybrid_ret.invoke(q3)[0].metadata['id'])
print("محتوى الوثيقة:", hybrid_ret.invoke(q3)[0].page_content)


السؤال: 'cheap noise canceling headphones'
الهجين قدر يجمع بين المترادفات والحروف وطلع الوثيقة: doc_ecommerce
محتوى الوثيقة: Affordable ANC wireless earbuds with active noise cancellation and clear mic.


---
### مساحتك للتجربة الحرة
جرب تسأل عن كود دالة الـ API: `"stream=True"` ولاحظ مين فيهم اللي هيعرف يجيبها أسرع وأدق.


In [5]:
# جرب هنا:
# my_q = "stream=True"
# print(hybrid_ret.invoke(my_q)[0].page_content)
